# 11 groupBy

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Group rows by one or more keys, understand what <code>groupBy</code> returns, and explain exactly what Spark does when you run <code>df.groupBy(...).count()</code>: partial aggregation, shuffle, final aggregation.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Grouping is the heart of reporting and of every Gold table: revenue per day, orders per customer, events per device. It is also the first operation most people meet that causes a <b>shuffle</b>. "What happens when I run <code>df.groupBy("x").count()</code>?" is one of the most common Spark interview questions. This lesson is <b>interview-critical</b>.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How groupBy works</b><br>
<code>df.groupBy("key")</code> does <b>not</b> return a DataFrame. It returns a <code>GroupedData</code> object that is waiting for an aggregation:
<ul><li><code>.count()</code>, <code>.sum("col")</code>, <code>.avg("col")</code>, <code>.min("col")</code>, <code>.max("col")</code> for one quick aggregate</li><li><code>.agg(...)</code> for several aggregates with names (lesson 12)</li><li><code>.pivot(...)</code> to turn values into columns (lesson 12)</li></ul>
The result has <b>one row per distinct key combination</b>. Rows with a null key form their own group.
</div>

```
groupBy is a WIDE transformation:

 Partition 1          Partition 2               after SHUFFLE (by key)          result
 IN 100               US 50                     -> IN: 100, 30, 70          ->  IN 200
 US 20                IN 30                     -> US: 20, 50               ->  US  70
 IN 70                UK 10                     -> UK: 10                   ->  UK  10
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An e-commerce company stores 500 million order lines. The daily sales dashboard needs revenue per country per day. Spark first sums each country <b>inside each partition</b> (so a partition with 2 million Indian orders sends one number, not 2 million rows), then shuffles those partial sums so each country's values meet on one executor, and finally adds them up. That partial step is why <code>groupBy().sum()</code> scales so well.
</div>

## Syntax

```python
from pyspark.sql import functions as F

df.groupBy("country").count()
df.groupBy("country", "category").sum("amount")
df.groupBy(F.year("order_date").alias("year")).agg(F.sum("amount").alias("revenue"))

# SQL
spark.sql("SELECT country, count(*) AS orders FROM orders GROUP BY country")
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an order lines DataFrame with countries, categories, dates and amounts, including a null country.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 10-row table with <code>order_id</code>, <code>country</code>, <code>category</code>, <code>order_date</code>, <code>amount</code> and <code>qty</code>.
</div>

In [0]:
from pyspark.sql import functions as F

orders = spark.createDataFrame(
    [
        (1, "IN", "Electronics", "2024-01-05", 1200.0, 1),
        (2, "IN", "Books", "2024-01-05", 25.0, 3),
        (3, "US", "Electronics", "2024-01-06", 800.0, 2),
        (4, "UK", "Books", "2024-01-06", 40.0, 1),
        (5, "IN", "Electronics", "2024-02-01", 300.0, 1),
        (6, "US", "Home", "2024-02-02", 150.0, 4),
        (7, "UK", "Home", "2024-02-03", 90.0, 2),
        (8, "US", "Books", "2024-02-03", 18.0, 1),
        (9, None, "Books", "2024-02-04", 12.0, 1),
        (10, "IN", "Home", "2024-02-05", 60.0, 2),
    ],
    "order_id INT, country STRING, category STRING, order_date STRING, amount DOUBLE, qty INT",
).withColumn("order_date", F.to_date("order_date"))
display(orders)

## 1. Group by one column

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Counts orders per country, then sums the amount per country.<br><br>
<b>Why it matters</b><br>The simplest grouping. Note the null country: it becomes its own group, not dropped.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 groups: IN (4 orders), US (3), UK (2) and NULL (1). Then the total amount per country, for example IN = 1585.0.
</div>

In [0]:
orders.groupBy("country").count().orderBy("country").show()

orders.groupBy("country").sum("amount").orderBy("country").show()

## 2. GroupedData is not a DataFrame

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the type returned by <code>groupBy</code>, and what happens if you try to <code>show()</code> it.<br><br>
<b>Why it matters</b><br>A common beginner error is to treat <code>groupBy</code> as if it already returned data. It only becomes a DataFrame after an aggregation.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>GroupedData</code> printed as the type, then an <code>AttributeError</code> because <code>GroupedData</code> has no <code>show</code> method.
</div>

In [0]:
grouped = orders.groupBy("country")
print(type(grouped).__name__)

try:
    grouped.show()
except AttributeError as e:
    print("AttributeError:", e)

## 3. Group by several columns and by expressions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Groups by <code>country</code> and <code>category</code>, then by a derived month column.<br><br>
<b>Why it matters</b><br>Real reports group on multiple dimensions, and often on a computed key such as a month, a week or a band.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row per (country, category) pair with summed amounts, then one row per month (<code>2024-01</code> and <code>2024-02</code>) with order counts 4 and 6.
</div>

In [0]:
orders.groupBy("country", "category").sum("amount", "qty").orderBy("country", "category").show()

orders.groupBy(F.date_format("order_date", "yyyy-MM").alias("month")).count().orderBy("month").show()

## 4. Quick aggregate methods

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses the shortcut methods <code>avg</code>, <code>min</code>, <code>max</code> and <code>sum</code> on <code>GroupedData</code>, and passes several columns at once.<br><br>
<b>Why it matters</b><br>Shortcuts are fine for exploration. For production you will mostly use <code>agg()</code> with aliases (lesson 12), because the shortcut column names like <code>avg(amount)</code> are awkward.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Average, minimum and maximum amount per category, with auto-generated names like <code>avg(amount)</code>.
</div>

In [0]:
orders.groupBy("category").avg("amount").show()
orders.groupBy("category").min("amount").show()
orders.groupBy("category").max("amount", "qty").show()

## 5. Filter after grouping (HAVING)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Keeps only the countries with more than 2 orders, which is the SQL <code>HAVING</code> clause.<br><br>
<b>Why it matters</b><br>Filters <b>before</b> a groupBy reduce the rows going into it (WHERE). Filters <b>after</b> work on the aggregated results (HAVING). Mixing them up is a classic SQL interview question.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The DataFrame version and the SQL version both return IN (4) and US (3).
</div>

In [0]:
(
    orders.groupBy("country")
    .agg(F.count("*").alias("orders"))
    .filter(F.col("orders") > 2)           # HAVING
    .orderBy("country")
    .show()
)

orders.createOrReplaceTempView("orders_v")
spark.sql("""
    SELECT country, count(*) AS orders
    FROM orders_v
    WHERE amount > 0            -- filter rows before grouping
    GROUP BY country
    HAVING count(*) > 2         -- filter groups after grouping
    ORDER BY country
""").show()

## 6. Edge cases: nulls in keys and values, empty groups

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows how <code>count("*")</code>, <code>count(col)</code> and <code>sum</code> behave with nulls, and that <code>groupBy()</code> with no keys aggregates the whole DataFrame.<br><br>
<b>Why it matters</b><br><code>count(col)</code> ignores nulls but <code>count("*")</code> does not. That difference produces "why don't my numbers match?" bugs.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For group <code>A</code>: 3 rows, 2 non-null values, sum 30. For the null key group: 1 row. Then the grand totals for the whole DataFrame in one row.
</div>

In [0]:
vals = spark.createDataFrame(
    [("A", 10), ("A", 20), ("A", None), (None, 5)], "grp STRING, v INT"
)

vals.groupBy("grp").agg(
    F.count("*").alias("rows"),
    F.count("v").alias("non_null_v"),
    F.sum("v").alias("sum_v"),
).orderBy("grp").show()

# groupBy() with no columns = one group containing every row (same as df.agg(...))
vals.groupBy().agg(F.count("*").alias("rows"), F.sum("v").alias("total")).show()

## Under the hood: what happens when you run `groupBy().count()`

```
orders.groupBy("country").count()

Stage 1 (map side, one task per input partition)
  HashAggregate(keys=[country], functions=[partial_count(1)])
     -> each task counts rows per country INSIDE its own partition
     -> writes small shuffle files: (country, partial_count)

        ========  Exchange hashpartitioning(country, N)  ========   <- SHUFFLE
        each row goes to partition  hash(country) % N
        so all partial counts for one country land in the same task

Stage 2 (reduce side, one task per shuffle partition)
  HashAggregate(keys=[country], functions=[count(1)])
     -> adds the partial counts to get the final count per country
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> `groupBy` + aggregation needs a shuffle (an `Exchange`), so it splits the job into two stages.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> nothing happens until `show()`, `count()`, `write`, and so on.

**Key points to say in an interview:**
- The partial aggregate (like a MapReduce *combiner*) shrinks data before the shuffle
- The shuffle is the expensive part: data is written to disk, sent over the network and read back
- The number of reduce tasks is `spark.sql.shuffle.partitions` (default 200), which **Adaptive Query Execution** (AQE) then coalesces into fewer partitions when they are small. On serverless this is managed for you

Run the cell below and find the two `HashAggregate` nodes and the `Exchange` between them.

In [0]:
(
    spark.range(5_000_000)
    .withColumn("country", (F.col("id") % 5).cast("string"))
    .groupBy("country")
    .count()
).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>'GroupedData' object has no attribute 'show'</code></b><br>
<code>groupBy</code> needs an aggregation before you can show or write the result.<br><br>
**⛔ Problem: totals don't match between <code>count("*")</code> and <code>count(col)</code>**<br><code>count(col)</code> skips nulls. Decide which one the business question needs.<br><br>
<b>⛔ Problem: unexpected null group</b><br>Rows with a null key form their own group. Filter or <code>coalesce</code> the key first if needed.<br><br>
<b>⛔ Problem: grouping is very slow and one task runs forever</b><br>Data skew: one key has far more rows than others, so one reduce task does most of the work. AQE skew handling, salting and pre-aggregation are covered in <code>05_optimization</code>.<br><br>
**⛔ Problem: <code>[MISSING_AGGREGATION]</code> in SQL**<br>A selected column is neither in <code>GROUP BY</code> nor wrapped in an aggregate. Add it to the grouping or aggregate it.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What happens internally when you run <code>df.groupBy("country").count()</code>?</b><br>
Spark plans two stages. In the first, each task does a partial count per country inside its own partition. Then an Exchange shuffles those partial results so all rows with the same country hash to the same partition. In the second stage, each task adds up the partial counts to produce the final result. Nothing runs until an action is called.<br><br>

<b>🎤 2. Is <code>groupBy</code> a narrow or wide transformation?</b><br>
Wide, because rows with the same key can be on any partition and must be brought together with a shuffle.<br><br>

<b>🎤 3. Why is the partial aggregation important?</b><br>
It reduces the data before the shuffle. Instead of sending every row across the network, each partition sends one row per key. That is why <code>groupBy().sum()</code> scales much better than collecting all values per key.<br><br>

<b>🎤 4. How many partitions does the result of a <code>groupBy</code> have?</b><br>
Initially <code>spark.sql.shuffle.partitions</code>, which defaults to 200. With Adaptive Query Execution, Spark coalesces small shuffle partitions at runtime, so the actual number is often much lower. On Databricks serverless it is tuned automatically.<br><br>

<b>🎤 5. What is the difference between WHERE and HAVING?</b><br>
WHERE filters rows before grouping. HAVING filters groups after aggregation. In PySpark both are <code>filter</code>, placed before or after the <code>groupBy</code>.<br><br>

<b>🎤 6. How are nulls handled in <code>groupBy</code>?</b><br>
Null keys form their own group. Inside aggregates, <code>count(col)</code>, <code>sum</code>, <code>avg</code>, <code>min</code> and <code>max</code> ignore null values, while <code>count(*)</code> counts every row.<br><br>

<b>🎤 7. What causes a groupBy to be slow?</b><br>
Usually a large shuffle or data skew, where one key holds a big share of the rows so one task does most of the work. You fix it by filtering early, selecting fewer columns, letting AQE handle skew, or salting the key.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which operation causes a shuffle?</b><br>
A. <code>df.filter(col("x") &gt; 1)</code><br>
B. <code>df.select("a", "b")</code><br>
C. <code>df.groupBy("a").count()</code><br>
D. <code>df.withColumn("c", lit(1))</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> Grouping needs rows with the same key on the same partition. The others are narrow.</details><br><br>

<b>Q2. A SQL query must return only departments whose average salary is above 50,000. Which clause filters on the average?</b><br>
A. <code>WHERE avg(salary) &gt; 50000</code><br>
B. <code>HAVING avg(salary) &gt; 50000</code><br>
C. <code>QUALIFY avg(salary) &gt; 50000</code><br>
D. <code>FILTER avg(salary) &gt; 50000</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> Conditions on aggregates go in <code>HAVING</code>. <code>QUALIFY</code> filters on window function results.</details><br><br>

<b>Q3. A table has 1,000 rows, of which 100 have a NULL <code>email</code>. What does <code>SELECT count(email) FROM t</code> return?</b><br>
A. 1000<br>
B. 900<br>
C. 100<br>
D. NULL<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>count(column)</code> ignores NULLs. <code>count(*)</code> would return 1000.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>orders</code>:
<ol><li>Count orders per <code>category</code></li><li>Total <code>qty</code> per <code>country</code> and <code>category</code>, sorted by country then category</li><li>Revenue (sum of <code>amount</code>) per month, using <code>date_format</code></li><li>Categories with total revenue above 500 (HAVING), in both PySpark and SQL</li><li>Number of orders with a known country versus an unknown one, in one groupBy (hint: group on an expression)</li><li>In two or three sentences, explain to an interviewer what happens when task 1 runs</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Orders per category
orders.groupBy("category").count().show()

# 2. Two grouping keys
orders.groupBy("country", "category").sum("qty").orderBy("country", "category").show()

# 3. Revenue per month
orders.groupBy(F.date_format("order_date", "yyyy-MM").alias("month")).agg(
    F.sum("amount").alias("revenue")
).orderBy("month").show()

# 4. HAVING, both ways
orders.groupBy("category").agg(F.sum("amount").alias("revenue")).filter("revenue > 500").show()
spark.sql("SELECT category, sum(amount) AS revenue FROM orders_v GROUP BY category HAVING sum(amount) > 500").show()

# 5. Group on an expression
orders.groupBy(F.col("country").isNull().alias("country_unknown")).count().show()

# 6. Each partition counts rows per category locally (partial aggregate), the partial counts are
#    shuffled so each category lands in one partition, then a final aggregate sums them.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>groupBy</code> returns <code>GroupedData</code>, which becomes a DataFrame only after an aggregation</li><li>One output row per distinct key combination. Null keys form their own group</li><li>Wide transformation: partial aggregate, then shuffle (<code>Exchange</code>), then final aggregate</li><li>Shuffle partitions start at <code>spark.sql.shuffle.partitions</code> (200) and AQE coalesces them</li><li>WHERE = filter before grouping, HAVING = filter after</li><li><code>count("*")</code> counts rows, <code>count(col)</code> skips nulls</li><li>Slow groupBy usually means a big shuffle or skewed keys</li></ul>
</div>

| Task | Code |
|---|---|
| Count per key | `df.groupBy("k").count()` |
| Several keys | `df.groupBy("a", "b").sum("x")` |
| Expression key | `df.groupBy(F.year("d").alias("y"))` |
| HAVING | `df.groupBy("k").agg(F.sum("x").alias("t")).filter("t > 10")` |
| Whole table | `df.groupBy().agg(...)` or `df.agg(...)` |

## Git commit

```
git add 01_spark_basics/11_groupby.ipynb
git commit -m "add 11 groupby notebook"
```